# E6 | Model Clustering K-Means
Segmentar incidentes em 4 clusters (A/B/C/D) para atuacao preventiva

In [28]:
import warnings
warnings.filterwarnings('ignore')
import os, pandas as pd, numpy as np, tempfile, joblib
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import mlflow

load_dotenv()
print('Setup OK')

Setup OK


In [29]:
RDS_HOST     = os.getenv('RDS_HOST')
RDS_PORT     = int(os.getenv('RDS_PORT', '5432'))
RDS_USER     = os.getenv('RDS_USER', 'postgres')
RDS_PASSWORD = os.getenv('RDS_PASSWORD')
RDS_DATABASE = os.getenv('RDS_DATABASE', 'aiops_gold')

print(f'Host: {RDS_HOST}')
print(f'DB:   {RDS_DATABASE}')
print(f'User: {RDS_USER}')

connection_url = f"postgresql+psycopg2://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:{RDS_PORT}/{RDS_DATABASE}"

%load_ext sql
%sql {connection_url}

Host: terraform-20260518150028461700000001.c4xegmk24lg6.us-east-1.rds.amazonaws.com
DB:   aiops_gold
User: postgres
The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [30]:

from sqlalchemy import create_engine
# Reutilizar credenciais da célula anterior (RDS_HOST, RDS_USER, RDS_PASSWORD, RDS_DATABASE)
engine = create_engine(f"postgresql://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:{RDS_PORT}/{RDS_DATABASE}")

df = pd.read_sql('''SELECT * FROM gold_ml.ml_cluster_dataset''',engine)

In [31]:
# ===== [4a] ESTRATÉGIA DE OUTLIERS MELHORADA (OR lógico) =====
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA

# Focar em features numéricas para detectar outliers
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['incident_id', 'cluster', 'data_abertura']]

print(f'📊 Dataset Original: {len(df):,} incidentes')
print(f'   Colunas numéricas para outlier detection: {len(numeric_cols)}')

# Calcular quantas colunas são outlier por linha (estratégia OR)
outlier_counts = pd.DataFrame(index=df.index)
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    # Marca como outlier se sai do intervalo
    outlier_counts[col] = ~((df[col] >= lower) & (df[col] <= upper))

# Remover apenas se MAIS DE METADE das colunas for outlier
outlier_total_per_row = outlier_counts.sum(axis=1)
max_outliers_allowed = len(numeric_cols) * 0.5
outlier_mask = outlier_total_per_row <= max_outliers_allowed

df_clean = df[outlier_mask].copy()
n_removed = len(df) - len(df_clean)

print(f'\n🔍 Outlier Detection (IQR — OR lógico, threshold={max_outliers_allowed:.0f} colunas):')
print(f'   Removidos: {n_removed:,} incidentes ({n_removed/len(df)*100:.2f}%)')
print(f'   Mantidos: {len(df_clean):,} incidentes ({len(df_clean)/len(df)*100:.2f}%)')

# Preparar features para o dataset limpo
feature_cols = [c for c in df_clean.columns if c not in ['incident_id', 'cluster', 'data_abertura']]
X_clean = df_clean[feature_cols].fillna(0)

# Identificar colunas categóricas e numéricas
cat_cols = X_clean.select_dtypes(include=['object']).columns.tolist()
num_cols = X_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f'\n📈 Features:')
print(f'   Colunas categóricas: {cat_cols}')
print(f'   Colunas numéricas: {num_cols}')

# Converter categóricas para string e fazer OneHotEncoding
X_processed = X_clean.copy()
if cat_cols:
    for col in cat_cols:
        X_processed[col] = X_processed[col].astype(str)
    
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    X_cat_encoded = encoder.fit_transform(X_processed[cat_cols])
    X_cat_df = pd.DataFrame(X_cat_encoded, columns=encoder.get_feature_names_out(cat_cols), index=X_processed.index)
    X_processed = pd.concat([X_processed[num_cols].reset_index(drop=True), X_cat_df.reset_index(drop=True)], axis=1)

# Normalizar com StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_processed)

print(f'\n✅ Features após OneHotEncoding: {X_processed.shape[1]}')
print(f'   Shape escalado: {X_scaled.shape}')

📊 Dataset Original: 121,811 incidentes
   Colunas numéricas para outlier detection: 17

🔍 Outlier Detection (IQR — OR lógico, threshold=8 colunas):
   Removidos: 60 incidentes (0.05%)
   Mantidos: 121,751 incidentes (99.95%)

📈 Features:
   Colunas categóricas: ['grupo_designado', 'categoria', 'subcategoria', 'produto', 'turno_abertura']
   Colunas numéricas: ['prioridade_num', 'hora_abertura', 'dia_semana_num', 'fora_horario_comercial', 'abriu_fim_de_semana', 'mes_abertura', 'trimestre', 'possui_pai', 'is_filho_de_problema', 'triagem_incompleta', 'duracao_horas', 'horas_ate_resolucao', 'foi_resolvido', 'excedeu_tempo_esperado', 'fechado_sem_tecnico', 'target_risco_sla', 'score_risco_operacional', 'duracao_horas_scaled', 'cluster_id']

✅ Features após OneHotEncoding: 680
   Shape escalado: (121751, 680)


In [ ]:
# ===== [4c] PCA: REDUÇÃO DIMENSIONAL (95% variância) =====
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_scaled)
n_components = pca.n_components_
explained_var = sum(pca.explained_variance_ratio_)

print(f'\n📉 PCA: REDUÇÃO DIMENSIONAL')
print(f'   Features antes: {X_scaled.shape[1]}')
print(f'   Componentes após: {n_components}')
print(f'   Variância explicada: {explained_var*100:.2f}%')
print(f'   Shape pós-PCA: {X_pca.shape}')

In [32]:
# ===== [5] K-MEANS TRAINING (no espaço PCA) =====
k = 4

# Verificar se PCA foi executado (célula 4c)
if 'X_pca' not in locals():
    print('⚠️ Aviso: X_pca não definido. Usando X_scaled diretamente.')
    X_train = X_scaled
    n_components = X_scaled.shape[1]
else:
    X_train = X_pca
    if 'n_components' not in locals():
        n_components = X_pca.shape[1]

model = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
print(f'\n🤖 Treinando K-Means com k={k} (espaço com {n_components} dimensões)...')

try:
    labels = model.fit_predict(X_train)
    
    sil_score = silhouette_score(X_train, labels)
    db_score = davies_bouldin_score(X_train, labels)
    
    print(f'\n📊 Métricas de Clustering:')
    print(f'   Silhouette Score: {sil_score:.4f}')
    print(f'   Davies-Bouldin Index: {db_score:.4f}')
    print(f'   Dataset: {len(labels):,} amostras')
    print(f'   ✅ Modelo KMeans treinado com sucesso')
except Exception as e:
    print(f'❌ Erro ao treinar KMeans: {str(e)}')
    print(f'   Verifique se as células anteriores foram executadas corretamente')
    raise

⚠️ Aviso: X_pca não definido. Usando X_scaled diretamente.

🤖 Treinando K-Means com k=4 (espaço com 680 dimensões)...


KeyboardInterrupt: 

In [ ]:
# ===== [6a] ATRIBUIÇÃO DE CLUSTERS (limpos + outliers) =====
# Adicionar clusters ao dataset limpo
df_clean['cluster_pred'] = labels
cluster_names = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}
df_clean['cluster_label'] = df_clean['cluster_pred'].map(cluster_names)

print(f'\n✅ Clusters atribuídos ao dataset limpo ({len(df_clean):,} registros)')

# ===== PREDIZER CLUSTERS PARA OS OUTLIERS REMOVIDOS =====
df_outliers = df[~outlier_mask].copy() if 'outlier_mask' in locals() else pd.DataFrame()

if len(df_outliers) > 0:
    print(f'\n🔄 Atribuindo clusters aos {len(df_outliers):,} incidentes removidos como outliers...')
    
    try:
        # Preparar features dos outliers (mesmo processo que df_clean)
        feature_cols_outliers = [c for c in df_outliers.columns if c not in ['incident_id', 'cluster', 'data_abertura']]
        X_outliers = df_outliers[feature_cols_outliers].fillna(0)
        
        # OneHotEncoding com mesmo encoder
        X_outliers_proc = X_outliers.copy()
        if cat_cols and 'encoder' in locals():
            for col in cat_cols:
                if col in X_outliers_proc.columns:
                    X_outliers_proc[col] = X_outliers_proc[col].astype(str)
            
            X_outliers_cat = encoder.transform(X_outliers_proc[cat_cols])
            X_outliers_cat_df = pd.DataFrame(X_outliers_cat, columns=encoder.get_feature_names_out(cat_cols), index=X_outliers_proc.index)
            num_cols_o = [c for c in num_cols if c in X_outliers_proc.columns]
            X_outliers_proc = pd.concat([X_outliers_proc[num_cols_o].reset_index(drop=True), X_outliers_cat_df.reset_index(drop=True)], axis=1)
        
        # Escalar com mesmo scaler
        if 'scaler' in locals():
            X_outliers_scaled = scaler.transform(X_outliers_proc)
        else:
            X_outliers_scaled = X_outliers_proc
        
        # Aplicar PCA com mesmo transformer
        if 'pca' in locals() and X_train is X_pca:
            X_outliers_pca = pca.transform(X_outliers_scaled)
            labels_outliers = model.predict(X_outliers_pca)
        else:
            labels_outliers = model.predict(X_outliers_scaled)
        
        df_outliers['cluster_pred'] = labels_outliers
        df_outliers['cluster_label'] = df_outliers['cluster_pred'].map(cluster_names)
        
        print(f'   ✅ Clusters preditos para outliers')
    except Exception as e:
        print(f'   ⚠️ Erro ao predizer clusters para outliers: {str(e)}')
        print(f'   Continuando apenas com dataset limpo')
        df_outliers = pd.DataFrame()
else:
    print('   ℹ️ Nenhum outlier removido, pulando etapa')

# ===== COMBINAR TODOS OS INCIDENTES =====
if len(df_outliers) > 0:
    df_final = pd.concat([df_clean, df_outliers], ignore_index=False).sort_index()
else:
    df_final = df_clean.copy()

print(f'\n📋 Distribuição Final de Clusters:')
cluster_dist = df_final['cluster_label'].value_counts().sort_index()
for label in ['A', 'B', 'C', 'D']:
    count = cluster_dist.get(label, 0)
    pct = count / len(df_final) * 100 if count > 0 else 0
    print(f'   Cluster {label}: {count:>8,} incidentes ({pct:>5.2f}%)')

print(f'\n✅ Dataset Final:')
print(f'   Total: {len(df_final):,} incidentes (100.00%)')
print(f'   Todos os incidentes receberam cluster label')

In [ ]:
# ===== [7] SALVAR RESULTADOS (todos os incidentes) =====
from pathlib import Path

base_path = Path(r'D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\kmeans')
base_path.mkdir(parents=True, exist_ok=True)

print(f'\n💾 Salvando resultados em: {base_path}')

# Salvar atribuição de clusters para TODOS os incidentes
if 'df_final' in locals() and len(df_final) > 0:
    cluster_results = df_final[['incident_id', 'cluster_pred', 'cluster_label']].copy()
    cluster_results.to_csv(str(base_path / 'kmeans_cluster_assignments.csv'), index=False)
    print(f'   ✅ kmeans_cluster_assignments.csv ({len(cluster_results):,} registros)')

    # Salvar estatísticas por cluster das colunas numéricas
    num_cols_stats = [c for c in num_cols if c in df_final.columns] if 'num_cols' in locals() else []
    if num_cols_stats:
        profile_stats = df_final.groupby('cluster_label')[num_cols_stats].agg(['mean', 'std', 'min', 'max'])
        profile_stats.to_csv(str(base_path / 'kmeans_cluster_profile_stats.csv'))
        print(f'   ✅ kmeans_cluster_profile_stats.csv')

    # Resumo de métricas
    summary_data = {
        'métrica': [
            'Modelo',
            'N Clusters',
            'Silhouette Score',
            'Davies-Bouldin Index',
            'Componentes PCA',
            'Registros processados',
            'Registros removidos',
            'Total final'
        ],
        'valor': [
            'K-Means',
            k,
            f'{sil_score:.4f}' if 'sil_score' in locals() else 'N/A',
            f'{db_score:.4f}' if 'db_score' in locals() else 'N/A',
            n_components if 'n_components' in locals() else 'N/A',
            len(df_clean) if 'df_clean' in locals() else 0,
            len(df_outliers) if 'df_outliers' in locals() else 0,
            len(df_final)
        ]
    }
    summary = pd.DataFrame(summary_data)
    summary.to_csv(str(base_path / 'kmeans_summary.csv'), index=False)
    print(f'   ✅ kmeans_summary.csv')

    # Distribuição dos clusters (detalhado)
    cluster_dist = df_final['cluster_label'].value_counts().sort_index()
    dist_df = pd.DataFrame({
        'cluster': cluster_dist.index,
        'count': cluster_dist.values,
        'percentage': (cluster_dist.values / len(df_final) * 100).round(2)
    })
    dist_df.to_csv(str(base_path / 'kmeans_cluster_distribution.csv'), index=False)
    print(f'   ✅ kmeans_cluster_distribution.csv')

    print(f'\n✅ Todos os resultados salvos com sucesso!')
else:
    print('   ❌ Erro: df_final não definido ou vazio')

In [ ]:
# ===== [8a] ELBOW METHOD (ANTES do MLflow) =====
print('\n📊 Calculando Elbow Method (k=2 até k=7)...')
inertias = []
silhouettes = []
k_range = range(2, 8)

# Usar X_train (que é definido na célula 5 como X_pca ou X_scaled)
X_elbow = X_train if 'X_train' in locals() else (X_pca if 'X_pca' in locals() else X_scaled)

for k_test in k_range:
    km_test = KMeans(n_clusters=k_test, random_state=42, n_init=10, max_iter=300)
    km_test.fit(X_elbow)
    inertias.append(km_test.inertia_)
    sil_test = silhouette_score(X_elbow, km_test.labels_, sample_size=min(5000, len(X_elbow)))
    silhouettes.append(sil_test)

import matplotlib.pyplot as plt
fig_elbow, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig_elbow.suptitle('Elbow Method & Silhouette Score — Seleção de k ótimo', fontsize=13)

# Gráfico 1: Inertia
ax1.plot(k_range, inertias, 'bo-', linewidth=2, markersize=8, label='Inertia')
ax1.axvline(k, color='red', linestyle='--', linewidth=2, label=f'Selecionado k={k}')
ax1.set_xlabel('Número de Clusters (k)')
ax1.set_ylabel('Inertia (Within-Cluster Sum of Squares)')
ax1.set_title('Elbow Method')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Gráfico 2: Silhouette Score
ax2.plot(k_range, silhouettes, 'go-', linewidth=2, markersize=8, label='Silhouette')
ax2.axvline(k, color='red', linestyle='--', linewidth=2, label=f'Selecionado k={k}')
ax2.axhline(sil_score, color='red', linestyle=':', alpha=0.7, label=f'Score={sil_score:.4f}')
ax2.set_xlabel('Número de Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score por k')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
elbow_path = os.path.join(tempfile.gettempdir(), 'elbow_method.png')
fig_elbow.savefig(elbow_path, dpi=100, bbox_inches='tight')
plt.close(fig_elbow)
print('✅ Elbow method calculado')

# ===== [8b] RASTREAMENTO COMPLETO COM MLFLOW =====
mlflow.set_experiment('kmeans_clustering')
with mlflow.start_run(run_name="kmeans_v2_pca_k4_balanced"):
    
    # ===== PARÂMETROS DO MODELO =====
    mlflow.log_params({
        'model_type': 'KMeans',
        'n_clusters': k,
        'random_state': 42,
        'n_init': 10,
        'max_iter': 300,
        'outlier_detection_method': 'IQR (OR lógico)',
        'outlier_removal_percentage': round(n_removed / len(df) * 100, 2) if n_removed > 0 else 0,
        'outlier_threshold_columns': round(len(numeric_cols) * 0.5, 1),
        'features_before_encoding': X_processed.shape[1],
        'features_before_pca': X_processed.shape[1],
        'features_after_pca': n_components,
        'pca_variance_threshold': 0.95,
        'records_cleaned': len(df_clean),
        'records_outliers': len(df_outliers) if len(df_outliers) > 0 else 0,
        'records_total': len(df_final),
        'normalization': 'StandardScaler'
    })

    # ===== MÉTRICAS DE QUALIDADE =====
    mlflow.log_metrics({
        'silhouette_score': float(sil_score),
        'davies_bouldin_index': float(db_score),
        'n_samples': len(df_final),
        'n_features_final': n_components,
        'n_features_original': X_processed.shape[1]
    })

    # Distribuição dos clusters
    for label in sorted(df_final['cluster_label'].unique()):
        count = (df_final['cluster_label'] == label).sum()
        percentage = count / len(df_final) * 100
        mlflow.log_metric(f'cluster_{label}_count', int(count))
        mlflow.log_metric(f'cluster_{label}_percentage', round(percentage, 2))

    # ===== ARTEFATOS: MODELO =====
    model_path = os.path.join(tempfile.gettempdir(), 'kmeans_model.pkl')
    joblib.dump(model, model_path)
    mlflow.log_artifact(model_path, 'model')

    # Salvar scaler
    scaler_path = os.path.join(tempfile.gettempdir(), 'standard_scaler.pkl')
    joblib.dump(scaler, scaler_path)
    mlflow.log_artifact(scaler_path, 'preprocessing')

    # Salvar PCA
    pca_model_path = os.path.join(tempfile.gettempdir(), 'pca_95_var.pkl')
    joblib.dump(pca, pca_model_path)
    mlflow.log_artifact(pca_model_path, 'preprocessing')

    # ===== ARTEFATOS: GRÁFICOS =====
    mlflow.log_artifact(elbow_path, 'evaluation')

    # PCA Visualization 2D
    pca_2d = PCA(n_components=2, random_state=42)
    X_2d = pca_2d.fit_transform(X_train)
    
    fig_pca, ax_pca = plt.subplots(figsize=(10, 8))
    colors_map = {'A': '#E53935', 'B': '#1E88E5', 'C': '#43A047', 'D': '#8E24AA'}
    
    for label in sorted(df_final['cluster_label'].unique()):
        mask = df_final['cluster_label'].values == label
        ax_pca.scatter(X_2d[mask, 0], X_2d[mask, 1], 
                      c=colors_map[label], label=f'Cluster {label}', 
                      alpha=0.6, s=30, edgecolors='k', linewidth=0.5)
    
    ax_pca.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} var)')
    ax_pca.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} var)')
    ax_pca.set_title(f'K-Means Clustering — PCA 2D (k={k}, Silhouette={sil_score:.4f})')
    ax_pca.legend()
    ax_pca.grid(True, alpha=0.3)
    
    pca_path = os.path.join(tempfile.gettempdir(), 'pca_clusters_2d.png')
    fig_pca.savefig(pca_path, dpi=100, bbox_inches='tight')
    mlflow.log_artifact(pca_path, 'evaluation')
    plt.close(fig_pca)

    # Distribuição dos clusters
    fig_dist, ax_dist = plt.subplots(figsize=(10, 5))
    cluster_counts_final = df_final['cluster_label'].value_counts().sort_index()
    bars = ax_dist.bar(cluster_counts_final.index, cluster_counts_final.values, 
                       color=[colors_map[c] for c in cluster_counts_final.index])
    ax_dist.set_xlabel('Cluster Label')
    ax_dist.set_ylabel('Number of Incidents')
    ax_dist.set_title(f'Cluster Distribution (k={k}, Silhouette={sil_score:.4f}, Total={len(df_final):,})')
    ax_dist.grid(True, alpha=0.3, axis='y')
    
    for bar, val in zip(bars, cluster_counts_final.values):
        pct = val/len(df_final)*100
        ax_dist.text(bar.get_x()+bar.get_width()/2, bar.get_height()+500,
                    f'{val:,}\n({pct:.1f}%)', ha='center', va='bottom', fontweight='bold')
    
    dist_path = os.path.join(tempfile.gettempdir(), 'cluster_distribution.png')
    fig_dist.savefig(dist_path, dpi=100, bbox_inches='tight')
    mlflow.log_artifact(dist_path, 'evaluation')
    plt.close(fig_dist)

    # ===== METADADOS E TAGS =====
    mlflow.set_tag('model_type', 'KMeans')
    mlflow.set_tag('task', 'Unsupervised_Clustering')
    mlflow.set_tag('data_version', f'{pd.Timestamp.now().date()}_gold_ml_dataset')
    mlflow.set_tag('notebook', '07_model_clustering_kmeans')
    mlflow.set_tag('evaluation_metrics', 'Silhouette_Score + Davies_Bouldin_Index')
    mlflow.set_tag('preprocessing', 'IQR_OR_lógico + OneHotEncoding + StandardScaler + PCA')
    mlflow.set_tag('outlier_handling', 'IQR_OR_method_with_prediction')
    mlflow.set_tag('dimensionality_reduction', 'PCA_95_variance')
    mlflow.set_tag('feature_count_original', str(X_processed.shape[1]))
    mlflow.set_tag('feature_count_pca', str(n_components))

    print(f'\n✅ MLflow Tracking Completo!')
    print(f'   Run name: kmeans_v2_pca_k4_balanced')
    print(f'   Métricas: Silhouette={sil_score:.4f}, Davies-Bouldin={db_score:.4f}')
    print(f'   Distribuição Final:')
    for label in sorted(df_final['cluster_label'].unique()):
        count = (df_final['cluster_label'] == label).sum()
        pct = count / len(df_final) * 100
        print(f'      Cluster {label}: {count:>8,} incidentes ({pct:>5.2f}%)')
    print(f'   Artefatos: Modelo KMeans, Scaler, PCA, Elbow Method, PCA 2D, Distribuição')
    print(f'   Tags: model_type, task, data_version, preprocessing, outlier_handling, etc')